# GNSS Interference Hotspots
### Is GPS interference random noise, or a geographic signature of active conflict?

**Data:** adsb.lol `globe_history` archive (ODbL), processed by our own NIC < 7 pipeline; GPSJAM.org daily hexes (J. Wiseman / ADS-B Exchange) for validation; UCDP GED & Candidate events; ACLED (when available); GFZ Kp index; OpenStreetMap military sites; Natural Earth borders.

**Hypotheses**
- **H1:** degraded-GNSS rates are strongly spatially clustered (global Moran's I > 0, compact LISA hotspots).
- **H2:** interference tracks *active armed conflict*, not tension. Taiwan Strait and Korea are the falsification test.
- **H3:** after controlling for traffic, fleet mix, receiver coverage, latitude and geomagnetic storms, proximity to conflict still predicts interference, and *stand-off* warfare (air/drone strikes, shelling, interceptions) more than ground fighting.

**How this notebook runs:** every heavy step is precomputed by `scripts/run_analysis.py` on the desktop. Outputs sync to this laptop through OneDrive, so by default the notebook *shows* those saved results. Set `RECOMPUTE = True` to recompute a window live; that needs the adsb.lol USB drive plugged in. The core spatial-statistics cells (weights, Moran's I, LISA) always run live on one day.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from gnss.analysis import spatial as sp
from gnss.analysis import viz
from gnss.analysis.config import load_analysis_config
from gnss.analysis.data import load_panel, observed, select_region
from gnss.analysis.pipeline import Run, saved_figures, saved_table

warnings.filterwarnings("ignore")
%matplotlib inline
viz.style()

cfg = load_analysis_config()           # finds the USB drive automatically (any drive letter)
SOURCE = os.environ.get("GNSS_SOURCE") or ("adsblol" if cfg.has_adsblol else "gpsjam")
WINDOW = os.environ.get("GNSS_WINDOW", "H")      # H = Spiderweb / Israel-Iran 12-day war
RECOMPUTE = os.environ.get("GNSS_RECOMPUTE", "0") == "1"

run = Run(cfg, SOURCE, show=True)
sp.warmup()                            # compiles esda's numba kernels now (~45 s once), not mid-talk
w = cfg.windows[WINDOW]
display(Markdown(
    f"**Source:** `{SOURCE}` · **data root:** `{cfg.paths.root}` "
    f"({'found' if cfg.has_adsblol else 'adsb.lol data not attached: GPSJAM / saved outputs only'})  \n"
    f"**Window {w.id}:** {w.name}, {w.start} → {w.end}, regions {', '.join(w.regions)} · "
    f"**RECOMPUTE** = {RECOMPUTE}"))


def step(method: str, pattern: str | None = None):
    """Recompute a pipeline step (figures appear as they are saved) or show its saved figures."""
    if RECOMPUTE:
        getattr(run, method)(w)
    elif pattern:
        saved_figures(cfg, SOURCE, WINDOW, pattern)

## 1. The data and the small-denominator problem
A hex-day's rate is `bad aircraft / all aircraft`. Many hexes see only a handful of aircraft, so 1/1 = 100% is noise. Every rate below uses hexes with ≥ 5 aircraft, plus Empirical Bayes smoothing.

In [ ]:
panel = load_panel(cfg, SOURCE, w.start, w.end)
r = cfg["rate"]
print(f"{len(panel):,} hex-days, {panel['hex'].nunique():,} hexes, {panel['date'].nunique()} days")
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(np.log10(panel["n"]), bins=60, color=viz.SERIES[0])
ax.axvline(np.log10(r["min_n"]), color=viz.INK2, ls=":")
ax.text(np.log10(r["min_n"]) + 0.03, ax.get_ylim()[1] * 0.9, f"n ≥ {r['min_n']} kept", color=viz.INK2)
ax.set_xlabel("Aircraft per hex-day (log10)")
ax.set_ylabel("Hex-days")
ax.set_title("Traffic per hex-day", loc="left", color=viz.INK)
plt.show()
print(f"{(panel['n'] < r['min_n']).mean():.0%} of hex-days have fewer than {r['min_n']} aircraft")

In [ ]:
step("eda", "raw_vs_eb_*.png")
saved_figures(cfg, SOURCE, WINDOW, "region_share_timeseries.png") if not RECOMPUTE else None

## 2. Validation: our NIC pipeline vs GPSJAM *(adsb.lol only)*
Same hex, same UTC day. GPSJAM's "bad" definition is undocumented; ours is NIC < 7 (14 CFR § 91.227) for at least half of an aircraft's time in the hex.

In [ ]:
step("validate", "validation_*.png")
try:
    display(saved_table(cfg, SOURCE, WINDOW, "validation_by_day.csv").describe().round(3))
except FileNotFoundError:
    display(Markdown("*No validation table for this source/window.*"))

## 3. Confounders removed before any statistics *(adsb.lol only)*
- **Manoeuvring:** steep banking masks the GPS antenna (rows with abs(roll) > 30° for most of their time).
- **ADS-B version 0:** these transponders encode integrity differently.
- **Faulty avionics:** an aircraft is treated as faulty only if it is low-NIC *where other aircraft were fine*. An aircraft that is degraded for its whole flight inside a jammed zone is treated as a jamming victim and kept.

In [ ]:
step("ablation", "ablation_*.png")

## 4. Spatial weights and global Moran's I (live, peak day)
Neighbours are H3 ring-1 contiguity among the hexes observed that day, row-standardized. Islands get ring-2 neighbours.

In [ ]:
REGION = os.environ.get("GNSS_REGION", w.regions[0])
bbox = cfg.regions[REGION]
peak = run.peak(w, panel, bbox)
day = observed(select_region(panel[panel["date"] == peak], bbox), r["min_n"])
hw = sp.h3_weights(day["hex"].to_numpy(), cfg["spatial"]["island_policy"])
day = day.set_index("hex").loc[hw.cells].reset_index()
day["eb_rate"] = sp.eb_rate(day["bad"], day["n"])
print(f"{REGION}, {peak:%Y-%m-%d}: {hw.w.n} hexes, mean {hw.w.mean_neighbors:.1f} neighbours, "
      f"{hw.n_islands} islands ({hw.n_dropped} dropped)")
gm = sp.global_moran(day["bad"], day["n"], hw, cfg["spatial"]["permutations"], cfg["spatial"]["seed"])
knn = sp.global_moran(day["bad"], day["n"], sp.knn_weights(hw.cells, cfg["spatial"]["knn_k"]),
                      cfg["spatial"]["permutations"], cfg["spatial"]["seed"])
pd.DataFrame({"H3 contiguity": gm, f"KNN (k={cfg['spatial']['knn_k']})": knn}).T.round(4)

In [ ]:
viz.moran_scatter(day["eb_rate"].to_numpy(), hw.w, f"Moran scatterplot · {REGION} · {peak:%Y-%m-%d}",
                  run.src_line);

## 5. Local clusters (LISA) with false-discovery-rate control (live, peak day)

In [ ]:
lm = sp.local_moran(day["bad"], day["n"], hw, cfg["spatial"]["permutations"], cfg["spatial"]["alpha"],
                    cfg["spatial"]["seed"])
display(pd.crosstab(lm["cluster"], lm["cluster_fdr"], margins=True))
viz.lisa_maps(lm, run.countries, bbox, f"LISA clusters · {REGION} · {peak:%Y-%m-%d}", run.src_line);

## 6. Over time: daily Moran's I and hotspot persistence

In [ ]:
step("esda", "moran_timeseries.png")
if not RECOMPUTE:
    saved_figures(cfg, SOURCE, WINDOW, "persistence_*.png")

## 7. Modifiable areal unit problem: does the answer depend on hex size?

In [ ]:
step("resolution", "resolution_*.png")

## 8. Multivariate model: what predicts interference?
One row per hex over the window. Outcome: logit of the EB-smoothed pooled rate (and, separately, persistence).
- **Drivers:** proximity to conflict events and to states at war, military airfields and naval bases, and (with ACLED) air/drone, shelling, ground and interception events plus the stand-off share.
- **Controls:** traffic, latitude, and for adsb.lol only: altitude mix, light-aircraft share, ADS-B v0 share, high-roll share, MLAT share (receiver coverage).

OLS residuals are spatially autocorrelated, so spatial lag and spatial error models are reported next to OLS.

In [ ]:
step("model", "model_coefs_*.png")
try:
    diag = saved_table(cfg, SOURCE, WINDOW, "model_diagnostics.json")
    display(diag[["n", "ols_r2", "moran_res_I", "ols_aic", "lag_aic", "err_aic", "lag_rho", "err_lambda",
                  "dropped"]].round(3))
except FileNotFoundError:
    display(Markdown("*No model outputs for this source/window yet.*"))

## 9. Day to day: hex fixed-effects panel
Within each hex, does interference rise on days with nearby conflict events (lagged 0–3 days) or geomagnetic storms? SEs are clustered by hex. With ACLED, events are split into stand-off, ground and interception.

In [ ]:
step("panel_fe")
try:
    pc = saved_table(cfg, SOURCE, WINDOW, "panel_coefs.csv")
    display(pc[~pc["term"].str.startswith("dow_")].round(5))
except FileNotFoundError:
    display(Markdown("*No panel outputs for this source/window yet.*"))

## 10. Event studies around the window's key dates

In [ ]:
step("events_study", "event_study_*.png")

## 11. All windows at a glance (saved outputs)
Median daily Moran's I per region and window. This is the H2 comparison: conflict regions vs the Taiwan / Korea / CONUS controls.

In [ ]:
rows = []
for wid in cfg.windows:
    d = cfg.outputs_dir / SOURCE / wid
    for f in d.glob("moran_daily_*.csv"):
        m = pd.read_csv(f)
        rows.append({"window": wid, "region": f.stem.replace("moran_daily_", ""),
                     "median_I": m["I"].median(), "days_sig": (m["p_sim"] < 0.05).mean(), "days": len(m)})
summary = pd.DataFrame(rows)
summary.pivot(index="region", columns="window", values="median_I").round(2) if len(summary) else summary

## 12. Limitations
- **No aircraft, no measurement.** Closed airspace (Ukraine, parts of Iran and Syria) is missing, not clean; missing hexes are never treated as zero.
- **Descriptive, not causal.** Clusters show where receivers were degraded, not where jammers are.
- **Altitude.** ADS-B sees interference at flight level; ground effects differ.
- **Coverage drift.** The adsb.lol feeder network grew from 2023 to 2026; models control for traffic and MLAT share.
- **Event data.** UCDP Candidate data currently ends 2026-08-31, and ACLED Research access has a time lag; a window past the end of coverage falls back or is truncated.
- **OSM military sites** are unevenly mapped.

**Attribution:** adsb.lol (ODbL) · GPSJAM.org / ADS-B Exchange · UCDP (Uppsala University) · ACLED (acleddata.com) · GFZ Potsdam (Kp, CC BY 4.0) · © OpenStreetMap contributors (ODbL) · Natural Earth.